# Task Statement 2.2: Implement Structured Error Responses for MCP Tools

## 📌 Executive Overview: Goal, Scope & Exam Importance

### Certification Domain & Weight
- **Domain**: Domain 2: Tool Design & MCP Integration
- **Domain Weight**: 20% of total CCAR-F exam score
- **Scenario Alignment**: Scenario 1 (Customer Support) & Scenario 4 (Developer Productivity)

### The Core Goal
To design and implement robust, structured error responses for Model Context Protocol (MCP) tools. An architect must master the **`isError: true`** protocol flag, classify failures into a machine-readable **error taxonomy (`transient`, `validation`, `business`, `permission`)**, supply explicit **retryability signals (`isRetryable`)**, and distinguish **valid empty search results** from true execution failures.

### Why This Matters in Production
When an external tool fails in production (e.g. database timeout, invalid account number, insufficient funds), returning a generic error string like `"Error: Operation failed"` or dumping an unhandled 50-line Python stack trace leaves Claude blind. The model cannot determine:
1. Is this a temporary network glitch that will resolve in 1 second?
2. Did I format the parameter incorrectly?
3. Is this a hard business policy violation that must be escalated to a human?
Without structured error metadata, agents either enter **infinite retry loops** on permanent validation errors or **give up prematurely** on recoverable transient network drops.

### Exam Importance
Task Statement 2.2 is heavily tested through failure recovery scenarios. The exam will test whether you understand how MCP signals errors via `isError: true`, how Claude uses `isRetryable` to choose between backoff retries and parameter corrections, and why treating zero-record database queries as errors is a critical anti-pattern.

---

## 📖 Deep Narrative: Why We Do What We Do

### 1. The Problem: The Generic Error Blindspot
Consider an agent calling `transfer_funds(from_acc="A", to_acc="B", amount=500.0)`.
- **Case 1 (Network Hiccup)**: The backend payment gateway times out.
  - *Naive Return*: `"Error: Failed"`.
  - *Claude's Reaction*: Claude assumes the transfer failed permanently and tells the customer: *"I am unable to transfer funds."* (SLA breached!).
- **Case 2 (Permanent Policy Breach)**: The customer has only \$100 in their account.
  - *Naive Return*: `"Error: Failed"`.
  - *Claude's Reaction*: Claude thinks it was a glitch, retries the exact same call 10 times, exhausts iteration limits, and crashes.
- **Case 3 (Valid Empty Search)**: A customer searches for orders placed in 2019. The database returns 0 rows.
  - *Naive Return*: `{"isError": True, "error": "No orders found"}`.
  - *Claude's Reaction*: Claude tells the customer: *"Our order database is experiencing technical downtime."* (Hallucination!).

### 2. The Architectural Solution: MCP Structured Error Taxonomy
In production, every MCP tool failure adheres to the **Standard Error Contract**:
```json
{
  "type": "tool_result",
  "tool_use_id": "toolu_01AbC...",
  "isError": true,
  "content": [
    {
      "type": "text",
      "text": "{\n  \"error_code\": \"ERR_TIMEOUT\",\n  \"category\": \"transient\",\n  \"isRetryable\": true,\n  \"retryAfterMs\": 1500,\n  \"message\": \"Payment gateway timed out. Safe to retry with same idempotency key.\"\n}"
    }
  ]
}
```

### 3. The 4 Production Error Categories
1. **`transient` (`isRetryable: true`)**:
   - Timeouts, rate limits (HTTP 429), temporary connection drops.
   - Claude's Action: Pauses with exponential backoff and re-invokes the tool.
2. **`validation` (`isRetryable: false`)**:
   - Parameter regex mismatch, missing required subfield, invalid date format.
   - Claude's Action: Re-formats the input arguments and calls the tool again with corrected parameters.
3. **`business` (`isRetryable: false`)**:
   - Policy limit exceeded, insufficient balance, account suspended.
   - Claude's Action: Halts execution immediately and explains the policy limitation cleanly to the user.
4. **`permission` (`isRetryable: false`)**:
   - Insufficient role privileges, missing auth token.
   - Claude's Action: Reroutes to human supervisor or informs user of permission boundary.

### 4. The Empty Result Rule
A database query that executes successfully and returns **zero matching rows** is a **SUCCESSFUL OPERATION**, not an error!
- Return: `isError: false` with `content: "[]"` or `content: "{\"matches\": 0, \"records\": []}"`.
- Never set `isError: true` on valid empty search results.

---

## 🔑 Key Technical Terminology Explained

- **`isError` Flag**: A boolean property on an Anthropic / MCP `tool_result` message indicating whether the execution encountered a failure.
- **Error Taxonomy**: A standardized set of categorical failure types (`transient`, `validation`, `business`, `permission`) that provide semantic clarity to the LLM.
- **`isRetryable` Signal**: An explicit boolean guiding Claude's decision to re-execute the identical tool call versus taking an alternative action.
- **Valid Empty Result**: A successful query execution that returned zero records; must never be reported as an error.
- **Exponential Backoff**: Increasing wait duration between subsequent retries (e.g. 1s $\rightarrow$ 2s $\rightarrow$ 4s) to avoid overwhelming recovering services.

---

## 🏗️ Architecture: MCP Structured Error Decision Flow

```mermaid
flowchart TD
    ToolFailure["Tool Execution Failure"] --> ClassifyError{"Classify Root Cause"}
    
    ClassifyError -->|"Network Timeout / 503"| Transient["category: transient<br>isRetryable: true<br>retryAfterMs: 1500"]
    ClassifyError -->|"Invalid Account Format"| Validation["category: validation<br>isRetryable: false<br>message: Expected ACC-XXXXX"]
    ClassifyError -->|"Over Daily Limit"| Business["category: business<br>isRetryable: false<br>message: Transfer exceeds limit"]
    ClassifyError -->|"403 Forbidden"| Permission["category: permission<br>isRetryable: false<br>message: Role admin required"]
    
    Transient --> MCPPayload["Format MCP tool_result:<br>isError: true<br>content: [structured error JSON]"]
    Validation --> MCPPayload
    Business --> MCPPayload
    Permission --> MCPPayload
    
    MCPPayload --> ClaudeRecovery{"Claude Recovery Loop"}
    ClaudeRecovery -->|"isRetryable: true"| BackoffRetry["Wait and Retry Tool Call"]
    ClaudeRecovery -->|"isRetryable: false & validation"| FixParam["Correct Parameter & Retry"]
    ClaudeRecovery -->|"isRetryable: false & business"| EscalateHuman["Explain Policy / Escalate"]
```

---

## ⚠️ Critical Exam Distinctions & Distractor Teardown

| Error Handling Pattern | Correct MCP Standard | Exam Trap to Reject | Why the Trap Fails |
|---|---|---|---|
| **Error Signaling Protocol** | Set `isError: true` on the `tool_result` and provide structured JSON error metadata. | **TRAP**: Throwing unhandled Python exceptions or passing raw HTTP 500 HTML error pages. | Unhandled exceptions crash the host application; raw HTML dumps pollute context with unparseable markup. |
| **Error Granularity** | Provide machine-readable categories (`category: "transient" \| "validation" \| "business"`). | **TRAP**: Returning uniform generic text strings: *"Error: Operation failed"*. | Claude cannot determine whether retrying will succeed or fail, leading to either infinite loops or premature surrender. |
| **Empty Search Results** | Return `isError: false` with empty list `[]` when a valid query finds 0 matches. | **TRAP**: Setting `isError: true` when a database query finds zero records. | Claude assumes the database is broken or offline and hallucinates system outages to the user. |
| **Retry Guidance** | Provide explicit `isRetryable: boolean` in the error JSON payload. | **TRAP**: Letting Claude guess retryability based on system prompt rules. | Model guesses unpredictably; explicit flags provide deterministic recovery behavior. |

---

## 💡 Architectural Decision Matrix: Rules of Thumb

| Failure Scenario | Category | `isRetryable` | Recommended Action |
|---|---|---|---|
| **API 504 Gateway Timeout** | `transient` | `true` | Back off exponentially and re-execute. |
| **Malformed Date String** | `validation` | `false` | Inspect format requirement, fix argument, re-execute. |
| **Credit Card Limit Exceeded** | `business` | `false` | Halt immediately, explain policy to customer. |
| **Token Expired / Forbidden** | `permission` | `false` | Route to human supervisor or request re-authentication. |
| **0 Matches for Filter** | None (Success) | N/A | Return empty array `[]` with `isError: false`. |

---

## 🧠 Claude 4.5 Extended Thinking & Parameter Standards

- **Error Recovery Reasoning**: When Claude receives `isError: true` with a structured payload, Extended Thinking allows the model to dissect the error category during its reasoning phase and formulate an intelligent recovery plan before emitting its next action.
- **Sampling Rules**: Omit `temperature` whenever thinking is active.
- **Thinking Continuity**: When handling error retries, assistant thinking blocks from previous failed turns must be preserved with their cryptographic signatures.

---

## 🛠️ Hands-on Workbook Guide: Following the Code Step-by-Step

When exploring [`ts_2_2_mcp_structured_errors.ipynb`](ts_2_2_mcp_structured_errors.ipynb):
1. **Step 1: Setup & Error Contract Classes**: Defines structured error schemas conforming to MCP standards.
2. **Step 2: Simulating Transient Failures**: Demonstrates handling network timeouts with `isRetryable: true` and backoff.
3. **Step 3: Simulating Validation & Business Failures**: Demonstrates how Claude corrects invalid regex arguments and halts on business limits.
4. **Step 4: The Zero-Record Search Demonstration**: Verifies that empty query results are treated as clean successes (`isError: false`).
5. **Step 5: Verification Assertions**: Verifies that 100% of errors are categorized and zero unhandled exceptions escape to the user.

In [ ]:
# Step 1: Environment & Setup
import json
import sys
from pathlib import Path
from typing import Any, Dict, Optional

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(" Environment initialized for TS 2.2 MCP Structured Errors.")

### 🚨 Step 2: Structured Error Helper & Tool Implementations
We implement `create_mcp_error` and tools demonstrating transient, validation, business, and empty result states.

In [ ]:
# Step 2: MCP Error Helper & Tools
def create_mcp_error(category: str, message: str, is_retryable: bool, details: Optional[Dict[str, Any]] = None) -> Dict[str, Any]:
    """Formats an MCP structured error response conforming to certification standards."""
    return {
        "isError": True,
        "errorCategory": category,  # 'transient' | 'validation' | 'business' | 'permission'
        "isRetryable": is_retryable,
        "message": message,
        "details": details or {},
    }

def query_inventory(sku: str, simulate_mode: Optional[str] = None) -> Dict[str, Any]:
    """Simulates warehouse inventory lookups with structured error handling."""
    if simulate_mode == "timeout":
        return create_mcp_error("transient", "Warehouse database connection timed out after 3000ms.", is_retryable=True)
    if not sku.startswith("SKU-"):
        return create_mcp_error("validation", f"Invalid SKU format '{sku}'. Expected format: SKU-XXXX.", is_retryable=False)
    if sku == "SKU-99999":
        # Valid empty result: NOT an error!
        return {"isError": False, "sku": sku, "stock_count": 0, "is_empty_result": True}
    return {"isError": False, "sku": sku, "stock_count": 42, "is_empty_result": False}

def issue_refund(customer_id: str, amount: float) -> Dict[str, Any]:
    """Simulates refund processing with business policy bounds."""
    if amount > 500.0:
        return create_mcp_error(
            "business",
            f"Refund amount ${amount:.2f} exceeds automated policy threshold ($500.00). Escalate to manager.",
            is_retryable=False,
            details={"customer_id": customer_id, "threshold": 500.0, "requested": amount}
        )
    return {"isError": False, "customer_id": customer_id, "refund_amount": amount, "status": "success"}

print(" Structured error engine ready.")

### 🧪 Step 3: Interactive Simulation
We test all 4 primary error/result categories:
1. **Transient Timeout**: Retryable.
2. **Malformed SKU**: Non-retryable validation error.
3. **Valid Empty Result**: Success with 0 items (isError: False).
4. **Policy Cap Breach**: Business rule violation.

In [ ]:
# Step 3: Simulation
# 1. Transient Error
res_transient = query_inventory("SKU-101", simulate_mode="timeout")
print("Transient Error Result:", res_transient)

# 2. Validation Error
res_validation = query_inventory("INVALID_FORMAT_SKU")
print("\nValidation Error Result:", res_validation)

# 3. Valid Empty Result (isError: False!)
res_empty = query_inventory("SKU-99999")
print("\nValid Empty Result:", res_empty)

# 4. Business Rule Violation
res_business = issue_refund("CUST-100", 650.0)
print("\nBusiness Rule Error Result:", res_business)

### 🔬 Step 4: Verification Assertions
We assert that:
1. Transient errors have `isRetryable == True`.
2. Validation errors have `isRetryable == False`.
3. Valid empty results have `isError == False`.
4. Business errors have `isRetryable == False` and include details.

In [ ]:
# Step 4: Verification Checks
assert res_transient["isError"] is True
assert res_transient["errorCategory"] == "transient"
assert res_transient["isRetryable"] is True

assert res_validation["isError"] is True
assert res_validation["errorCategory"] == "validation"
assert res_validation["isRetryable"] is False

assert res_empty["isError"] is False
assert res_empty["stock_count"] == 0
assert res_empty["is_empty_result"] is True

assert res_business["isError"] is True
assert res_business["errorCategory"] == "business"
assert res_business["isRetryable"] is False

print(" All verification assertions passed! MCP error taxonomy verified.")

### 💡 CCAR-F Exam Golden Rules
- **Rule 1**: Always return structured errors using `isError: true` with an explicit `errorCategory`.
- **Rule 2**: Mark `isRetryable: true` ONLY for transient infrastructure failures (timeouts, rate limits).
- **Rule 3**: Never return an error for valid empty query results (0 matches). Return `isError: false` and `stock_count: 0`.